In [1]:
from pathlib import Path
import numpy as np
import os
from utils.benchmarks import BENCHMARKS_TO_PATHS

if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    import os

    os.environ["MODIN_ENGINE"] = "ray"
    import ray

    ray.init(
        num_cpus=int(os.environ["MODIN_CPUS"]),
        runtime_env={"env_vars": {"__MODIN_AUTOIMPORT_PANDAS__": "1"}},
    )
    import modin.pandas as pd
else:
    import pandas as pd
import time

In [2]:
### cell 0 ###

benchmark_name = "feedback3-eda-hf-custom-trainer-sift"
train_df = pd.read_csv(
    Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input" / "feedback-prize-english-language-learning" / "train.csv"
)
test_df = pd.read_csv(
    Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input" / "feedback-prize-english-language-learning" / "test.csv"
)

factor = 10
train_df = pd.concat([train_df] * factor)
test_df = pd.concat([test_df] * factor)
train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 39110 entries, 0 to 3910
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   text_id      39110 non-null  object 
 1   full_text    39110 non-null  object 
 2   cohesion     39110 non-null  float64
 3   syntax       39110 non-null  float64
 4   vocabulary   39110 non-null  float64
 5   phraseology  39110 non-null  float64
 6   grammar      39110 non-null  float64
 7   conventions  39110 non-null  float64
dtypes: float64(6), object(2)
memory usage: 2.7+ MB


In [3]:
### cell 1 ###

train_df.head()

,text_id,full_text,cohesion,syntax,vocabulary,phraseology,grammar,conventions
0,0016926B079C,I think that students would benefit from learn...,3.5,3.5,3.0,3.0,4.0,3.0
1,0022683E9EA5,When a problem is a change you have to let it ...,2.5,2.5,3.0,2.0,2.0,2.5
2,00299B378633,"Dear, Principal\n\nIf u change the school poli...",3.0,3.5,3.0,3.0,3.0,2.5
3,003885A45F42,The best time in life is when you become yours...,4.5,4.5,4.5,4.5,4.0,5.0
4,0049B1DF5CCC,Small act of kindness can impact in other peop...,2.5,3.0,3.0,3.0,2.5,2.5


In [4]:
### cell 2 ###

test_df.head()

,text_id,full_text
0,0000C359D63E,when a person has no experience on a job their...
1,000BAD50D026,Do you think students would benefit from being...
2,00367BB2546B,"Thomas Jefferson once states that ""it is wonde..."
0,0000C359D63E,when a person has no experience on a job their...
1,000BAD50D026,Do you think students would benefit from being...


In [5]:
### cell 3 ###

len(train_df), len(test_df)

(39110, 30)

In [6]:
### cell 4 ###

LABEL_COLUMNS = [
    "cohesion",
    "syntax",
    "vocabulary",
    "phraseology",
    "grammar",
    "conventions",
]

In [7]:
### cell 5 ###

texts = train_df.sample(frac=1, random_state=420).head(4)

In [8]:
### cell 6 ###

train_df["total_score"] = train_df[LABEL_COLUMNS].sum(axis=1)
lowest_df = train_df.sort_values("total_score").head(4)

In [12]:
train_df[LABEL_COLUMNS]

,cohesion,syntax,vocabulary,phraseology,grammar,conventions
0,3.5,3.5,3.0,3.0,4.0,3.0
1,2.5,2.5,3.0,2.0,2.0,2.5
2,3.0,3.5,3.0,3.0,3.0,2.5
3,4.5,4.5,4.5,4.5,4.0,5.0
4,2.5,3.0,3.0,3.0,2.5,2.5
...,...,...,...,...,...,...
3906,2.5,3.0,3.0,3.5,2.5,2.5
3907,4.0,4.0,4.0,4.0,3.5,3.0
3908,2.5,3.0,3.0,3.0,3.5,3.0
3909,4.0,4.5,4.5,4.0,4.5,4.5


In [ ]:
### cell 7 ###

train_df["total_score"] = train_df[LABEL_COLUMNS].sum(axis=1)
highest_df = train_df.sort_values("total_score", ascending=False).head(4)

In [ ]:
### cell 8 ###

train_df["word_count"] = train_df.full_text.apply(lambda x: len(x.split()))

In [ ]:
### cell 9 ###

train_df["word_count"].mean()

In [ ]:
### cell 10 ###

train_df["word_count"].max()